In [0]:
from pyspark.sql.functions import col

df = spark.table("workspace.messy_ecommerce_db.bronze_orders")

for c in df.columns:
    df = df.withColumn(c, col(c).cast("string"))

df = df.toDF(*[c.strip().lower().replace(" ","_") for c in df.columns])

In [0]:
df.count()

103

Import Packages


In [0]:
from pyspark.sql.functions import col, regexp_replace, to_date, initcap, lower, when, coalesce


Data Load and Header Replace with Lower Case

In [0]:

df = df.toDF(*[c.strip().lower().replace(" ","_") for c in df.columns])

Fixing Price Columns


In [0]:
from pyspark.sql.functions import expr
df = df.withColumn("price", regexp_replace(col("price"),"[^0-9.]",""))
df = df.withColumn("price", expr("try_cast(price as double)"))

Fixing Quantity

In [0]:
df = df.withColumn("quantity", regexp_replace(col("quantity"),"[^0-9.]",""))
df = df.withColumn("quantity",col("quantity").cast("int"))

Fixing Date Format

Standardize Category

In [0]:
df = df.withColumn("category", initcap(lower(col("category"))))

Fixing Payment Methods


In [0]:
df = df.withColumn("payment_method",
    regexp_replace(col("payment_method"), "Cash on D", "Cash on Delivery")
)

Handling Missing Values


In [0]:
df = df.fillna({"quantity":1, "price":0.0})

Remove Invalid Records

In [0]:
df = df.filter((col("quantity")> 0) & (col("price") > 0))

Total

In [0]:
df = df.withColumn("total_price",col("quantity") * col("price"))

Data Quality

In [0]:
df = df.withColumn("is_valid", when((col("quantity")>0) & (col("price")>0), True).otherwise(False))


In [0]:
from pyspark.sql.functions import expr

df = df.withColumn("order_date", col("order_date").cast("string"))

df = df.withColumn(
    "order_date",
    expr("""
        coalesce(
            try_to_timestamp(order_date, 'M/d/yyyy'),
            try_to_timestamp(order_date, 'MM-dd-yyyy'),
            try_to_timestamp(order_date, 'MMM d yyyy')
        )
    """)
)

In [0]:
df.printSchema()

root
 |-- id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = false)
 |-- price: double (nullable = false)
 |-- payment_method: string (nullable = true)
 |-- status: string (nullable = true)
 |-- total: string (nullable = true)
 |-- total_price: double (nullable = false)
 |-- is_valid: boolean (nullable = false)



In [0]:

df.write.mode("overwrite").format("delta") \
    .saveAsTable("workspace.messy_ecommerce_db.silver_orders")